# Agência excessiva: o chamado falso que cancelou a banca

OWASP Top 10 para LLMs 2025 — **LLM06 Excessive Agency**

Simulador didático (`campus/`), sem rede e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [1]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, narrar  # noqa: E402
from campus.llm import SimLLM  # noqa: E402

print("Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.")

Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.


## Cenário
Um chamado de suporte traz um texto oculto (`display:none`) pedindo para cancelar a reserva R-310 — o auditório da banca de defesa do PPG-Saúde.

In [2]:
print(dados.RESERVAS_INICIAIS["R-310"])


{'sala': 'auditório', 'evento': 'Banca de defesa PPG-Saúde', 'quando': 'sexta 14h'}


## Sem defesa
O agente executa qualquer chamada de ferramenta que o modelo pedir, sem checar permissão.

In [3]:
amb_vulneravel = ferramentas.Ambiente()
resp = SimLLM().gerar([
    {"role": "system", "content": "Você é o agente do Agenda Lab."},
    {"role": "user", "content": "Trate o chamado a seguir.\n\n" + dados.CHAMADO_MALICIOSO},
])
ferramentas.executar_chamadas(resp, amb_vulneravel)
for e in amb_vulneravel.log:
    print(e)
assert "R-310" not in amb_vulneravel.reservas  # a banca foi cancelada


{'ferramenta': 'reservar_sala', 'args': {'sala': 'auditório', 'horario': 'sexta às 14h'}, 'status': 'executada', 'detalhe': 'criaria a reserva R-402 (auditório, sexta às 14h)'}
{'ferramenta': 'enviar_aviso', 'args': {'destino': 'suporte-externo@parceiro.example', 'assunto': 'Aviso', 'corpo': 'a lista de bolsistas'}, 'status': 'executada', 'detalhe': 'enviaria e-mail para suporte-externo@parceiro.example: a lista de bolsistas'}
{'ferramenta': 'cancelar_reserva', 'args': {'id': 'R-310'}, 'status': 'executada', 'detalhe': 'cancelaria a reserva R-310'}


## Com defesa
`defesas.politica_agenda()` só permite `consultar_matricula` e `reservar_sala` (com confirmação humana); `cancelar_reserva` nem está na lista.

In [4]:
amb_defendido = ferramentas.Ambiente()
ferramentas.executar_chamadas(resp, amb_defendido, defesas.politica_agenda())
for e in amb_defendido.log:
    print(e)
assert "R-310" in amb_defendido.reservas  # a banca continua marcada


{'ferramenta': 'reservar_sala', 'args': {'sala': 'auditório', 'horario': 'sexta às 14h'}, 'status': 'bloqueada', 'detalhe': 'privilégio: pendente de confirmação humana (não aprovada)'}
{'ferramenta': 'enviar_aviso', 'args': {'destino': 'suporte-externo@parceiro.example', 'assunto': 'Aviso', 'corpo': 'a lista de bolsistas'}, 'status': 'bloqueada', 'detalhe': 'privilégio: ferramenta não concedida a este agente'}
{'ferramenta': 'cancelar_reserva', 'args': {'id': 'R-310'}, 'status': 'bloqueada', 'detalhe': 'privilégio: ferramenta não concedida a este agente'}


## O que observar
- O modelo pediu exatamente a mesma ação nos dois casos: a injeção funcionou sobre ele do mesmo jeito. O que mudou foi o poder que o agente tinha para obedecer.
- Agência excessiva tem três formas (permissão, funcionalidade, autonomia demais); aqui é a permissão: `cancelar_reserva` nunca deveria estar disponível sem revisão.
- Menor privilégio por ferramenta é a defesa mais barata e mais eficaz deste risco.